In [26]:
from langgraph.graph import START,END,StateGraph
from typing import Annotated,TypedDict,Literal
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
import os
from dotenv import load_dotenv
from langgraph.graph.message import add_messages

load_dotenv(dotenv_path=".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not found"
    )

In [27]:

class AgentState(TypedDict):
    messages: Annotated[list,add_messages]

from langgraph.graph.message import add_messages as add_msg_fn

existing=[HumanMessage(content="Hello !")]

new=[AIMessage(content="Hi there")]

combined=add_msg_fn(existing,new)

for msg in combined:
    print(f"{type(msg).__name__}  : {msg.content}")

HumanMessage  : Hello !
AIMessage  : Hi there


In [28]:
@tool
def get_weather(city:str)->str:
    """Get the current weather for a given city.
    
    Args:
        city: The name of the city to get weather for.
        
    Returns:
        A string describing the current weather.
    """
    weather_data = {
        "paris": "Sunny, 22°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C",
    }

    city_lower=city.lower()
   
    if city_lower in weather_data: 
        return f"The weather for {city} is {weather_data[city_lower]}"
    else:
        return f"{city} not found"

print("Testing Tool:")
print(get_weather.invoke({"city":"Paris"}))
    

Testing Tool:
The weather for Paris is Sunny, 22°C


In [29]:
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.7)
tools=[get_weather]
llm_with_tools=llm.bind_tools(tools)

print(f"LLM configured with tools")

LLM configured with tools


In [30]:
from typing import Literal
from langgraph.types import Command

class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

def call_model(state:AgentState):
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    return {"messages":[response]}

def tool_calls(state:AgentState)->Command:
    message=state['messages']
    ai_message=message[-1]
    tool_calls=ai_message.tool_calls

    tool_messages=[]
    for tool_call in tool_calls:
        tool_name=tool_call["name"]
        tool_args=tool_call["args"]

        print(f"Calling tool: {tool_name}:{tool_args}")
        tool={t.name:t for t in tools}[tool_name]
        result=tool.invoke(tool_args)

        tool_messages.append(ToolMessage(
            content=result,
            tool_call_id=tool_call["id"]
        ))

    return {"messages":tool_messages}

def should_continue(state:AgentState)->Literal["tools","__end__"]:
    message=state["messages"]
    ai_message=message[-1]
    if ai_message.tool_calls:
        print(f"The LLM has requested Tool Calls")
        return "tools"
    else:
        print(f"No tool Call required ")
        return "__end__"




In [31]:
graph=StateGraph(AgentState)

graph.add_node("agent",call_model)
graph.add_node("tools",tool_calls)

graph.add_edge(START,"agent")
graph.add_conditional_edges("agent",should_continue)
graph.add_edge("tools","agent")

app=graph.compile()
print(f"Agent is ready")

Agent is ready


In [32]:
query = "What's the weather like in Paris?"

result=app.invoke({"messages":[HumanMessage(content=query)]})
response=result['messages'][-1].content

print(response)

The LLM has requested Tool Calls
Calling tool: get_weather:{'city': 'Paris'}
No tool Call required 
The weather in Paris is sunny with a temperature of 22°C.


In [ ]:
from langgraph.types import Command

class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

def call_model(state:AgentState)->Command[Literal["agent","__end__"]]:
    message=state['messages']
    
    result=llm_with_tools.invoke(message)


    return Command(update={"messages":[result]},goto="agent" if result.tool_calls else END)

def call_tool(state:AgentState)->Command[Literal["model"]]:
    message=state['messages']
    ai_message=message[-1]
    tool_calls=ai_message.tool_calls
    tool_message=[]

    for tool_call in tool_calls:
        tool_name=tool_call["name"]
        tool_arg=tool_call["args"]

        tool_obj={t.name:t for t in tools}[tool_name]
        result=tool_obj.invoke(tool_arg)

        tool_message.append(ToolMessage(content=result,tool_call_id=tool_call["id"]))

    return Command(update={"messages":tool_message},goto="model")

graph=StateGraph(AgentState)

graph.add_node("model",call_model)
graph.add_node("agent",call_tool)

graph.add_edge(START,"model")


app=graph.compile()

query="What's the weather like in Paris?"

result=app.invoke({"messages": [HumanMessage(content=query)]})
print(result["messages"][-1].content)

The weather in Paris is sunny with a temperature of 22°C.


Parallel Tool calls

In [42]:
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage

response=AIMessage(
    content="",
    tool_calls=[
        {"name":"get_weather","args":{"city":"Paris"},"id":"call_1"},
        {"name":"get_weather","args":{"city":"London"},"id":"call_2"},
        {"name":"get_weather","args":{"city":"Berlin"},"id":"call_3"}
    ]
)

for tc in response.tool_calls:
    print(f" -> {tc["name"]} : {tc["args"]}")

 -> get_weather : {'city': 'Paris'}
 -> get_weather : {'city': 'London'}
 -> get_weather : {'city': 'Berlin'}
